<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Timeline_Reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python-based timeline reconstruction program that combines file metadata and system log events, normalizes different timestamp formats, sorts all events chronologically, and identifies important activities occurring before, during, and after a simulated incident.

**Algorithm**

Create simulated file metadata and system log datasets.

Read timestamps from both sources.

Identify the different timestamp formats.

Convert all timestamps into a common datetime format.

Assign an evidence source to each record.

Combine the records into a single dataset.

Sort all events according to normalized timestamps.

Define the simulated incident
time period.

Classify events as Before Incident, During Incident, or After Incident.

Display the reconstructed forensic timeline.

Summarize important events surrounding the incident.

In [1]:
# ==============================================
# FORENSIC TIMELINE RECONSTRUCTION
# ==============================================

import pandas as pd

print("=" * 90)
print("                 FORENSIC TIMELINE RECONSTRUCTION")
print("=" * 90)

# ------------------------------------------------
# 1. Simulated File Metadata Events
# ------------------------------------------------

file_data = [
    ["25-08-2026 08:30:00",
     "File Metadata",
     "File Created",
     "C:/Documents/report.docx",
     "New document created"],

    ["2026/08/25 08:45:00",
     "File Metadata",
     "File Modified",
     "C:/Documents/report.docx",
     "Document modified"],

    ["25-08-2026 09:05:00",
     "File Metadata",
     "File Accessed",
     "C:/Documents/passwords.txt",
     "Sensitive file accessed"],

    ["2026-08-25 10:30:00",
     "File Metadata",
     "File Modified",
     "C:/Documents/report.docx",
     "File modified after incident"]
]

# ------------------------------------------------
# 2. Simulated System Log Events
# ------------------------------------------------

system_data = [
    ["2026-08-25 08:50:00",
     "System Log",
     "Login",
     "User: Alice",
     "Successful user login"],

    ["25/08/2026 09:10:00",
     "System Log",
     "USB Connected",
     "Device: USB-003",
     "Unrecognized USB device connected"],

    ["2026/08/25 09:20:00",
     "System Log",
     "Process Started",
     "powershell.exe",
     "PowerShell process executed"],

    ["2026-08-25 09:35:00",
     "System Log",
     "Network Connection",
     "192.168.1.50",
     "Outbound network connection detected"],

    ["25-08-2026 10:00:00",
     "System Log",
     "Logout",
     "User: Alice",
     "User logged out"]
]

# ------------------------------------------------
# 3. Create DataFrames
# ------------------------------------------------

file_df = pd.DataFrame(
    file_data,
    columns=[
        "Timestamp",
        "Evidence_Source",
        "Event_Type",
        "Affected_Object",
        "Description"
    ]
)

system_df = pd.DataFrame(
    system_data,
    columns=[
        "Timestamp",
        "Evidence_Source",
        "Event_Type",
        "Affected_Object",
        "Description"
    ]
)

# ------------------------------------------------
# 4. Normalize Different Timestamp Formats
# ------------------------------------------------

file_df["Timestamp"] = pd.to_datetime(
    file_df["Timestamp"],
    dayfirst=True,
    errors="coerce"
)

system_df["Timestamp"] = pd.to_datetime(
    system_df["Timestamp"],
    dayfirst=True,
    errors="coerce"
)

# ------------------------------------------------
# 5. Combine Evidence Sources
# ------------------------------------------------

timeline = pd.concat(
    [file_df, system_df],
    ignore_index=True
)

# ------------------------------------------------
# 6. Check Invalid Dates
# ------------------------------------------------

invalid = timeline[
    timeline["Timestamp"].isna()
]

if not invalid.empty:

    print("\nWarning: Invalid timestamps detected.")
    print(invalid)

# Remove invalid records
timeline = timeline.dropna(
    subset=["Timestamp"]
)

# ------------------------------------------------
# 7. Sort Chronologically
# ------------------------------------------------

timeline = timeline.sort_values(
    "Timestamp"
).reset_index(drop=True)

# ------------------------------------------------
# 8. Define Simulated Incident Period
# ------------------------------------------------

incident_start = pd.Timestamp(
    "2026-08-25 09:10:00"
)

incident_end = pd.Timestamp(
    "2026-08-25 09:35:00"
)

# ------------------------------------------------
# 9. Classify Timeline Events
# ------------------------------------------------

def classify_event(timestamp):

    if timestamp < incident_start:
        return "BEFORE INCIDENT"

    elif timestamp <= incident_end:
        return "DURING INCIDENT"

    else:
        return "AFTER INCIDENT"


timeline["Incident_Phase"] = timeline[
    "Timestamp"
].apply(classify_event)

# ------------------------------------------------
# 10. Display Complete Timeline
# ------------------------------------------------

print("\n" + "=" * 90)
print("                     RECONSTRUCTED TIMELINE")
print("=" * 90)

print(
    timeline[
        [
            "Timestamp",
            "Evidence_Source",
            "Event_Type",
            "Affected_Object",
            "Description",
            "Incident_Phase"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 11. Important Incident Events
# ------------------------------------------------

important_events = timeline[
    timeline["Incident_Phase"] != "BEFORE INCIDENT"
]

print("\n" + "=" * 90)
print("                     INCIDENT-RELATED EVENTS")
print("=" * 90)

print(
    important_events[
        [
            "Timestamp",
            "Evidence_Source",
            "Event_Type",
            "Affected_Object",
            "Incident_Phase"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 12. Summary
# ------------------------------------------------

print("\n" + "=" * 90)
print("                          SUMMARY")
print("=" * 90)

print(
    "Total Events      :",
    len(timeline)
)

print(
    "Before Incident   :",
    len(timeline[
        timeline["Incident_Phase"] == "BEFORE INCIDENT"
    ])
)

print(
    "During Incident   :",
    len(timeline[
        timeline["Incident_Phase"] == "DURING INCIDENT"
    ])
)

print(
    "After Incident    :",
    len(timeline[
        timeline["Incident_Phase"] == "AFTER INCIDENT"
    ])
)

# ------------------------------------------------
# 13. Save Timeline
# ------------------------------------------------

timeline.to_csv(
    "/content/forensic_timeline.csv",
    index=False
)

print(
    "\nTimeline saved as: "
    "/content/forensic_timeline.csv"
)

print("\nTimeline reconstruction completed.")
print("=" * 90)

                 FORENSIC TIMELINE RECONSTRUCTION

  Timestamp Evidence_Source       Event_Type           Affected_Object  \
1       NaT   File Metadata    File Modified  C:/Documents/report.docx   
3       NaT   File Metadata    File Modified  C:/Documents/report.docx   
5       NaT      System Log    USB Connected           Device: USB-003   
6       NaT      System Log  Process Started            powershell.exe   
8       NaT      System Log           Logout               User: Alice   

                         Description  
1                  Document modified  
3       File modified after incident  
5  Unrecognized USB device connected  
6        PowerShell process executed  
8                    User logged out  

                     RECONSTRUCTED TIMELINE
          Timestamp Evidence_Source         Event_Type            Affected_Object                          Description  Incident_Phase
2026-08-25 08:30:00   File Metadata       File Created   C:/Documents/report.docx         

/tmp/ipykernel_1019/3220048618.py:113: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  system_df["Timestamp"] = pd.to_datetime(


**Result**

The Python program successfully combined file metadata and system log evidence into a single chronological timeline. Different timestamp formats were normalized into a common date-time format before sorting. The reconstructed timeline clearly identified events occurring before, during, and after the simulated incident, including the unrecognized USB connection, PowerShell execution, and network connection during the incident period. This provides investigators with a unified sequence for reconstructing the possible attack or incident progression.